# 03 - Avaliacao

Grade comparativa (base x config1 x config2), CLIPScore, checagem de memorizacao e preparo do formulario cego.

As 3 primeiras secoes (instalar, carregar base, prompts) rodam sem precisar do treino terminado. As secoes de geracao precisam dos 2 LoRAs ja publicados no Hub.

## 1. Instalar dependencias

In [ ]:
!pip -q install diffusers transformers accelerate peft torchmetrics matplotlib

## 1.5 Clonar o dataset

Necessário pra seção 7 (verificação de memorização), que compara imagens geradas com as originais do dataset.

In [ ]:
!git clone https://github.com/lramosc1512/atelie-generativo-LeonidIA.git
%cd /content

## 2. Carregar o modelo base

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
pipe.set_progress_bar_config(disable=True)
print("Modelo base carregado.")

## 3. Prompts fixos de teste

6 prompts cobrindo prédios diferentes do dataset, com a mesma seed pra cada um em todos os modelos - assim a comparação isola o efeito do LoRA, não do prompt/seed.

In [ ]:
PROMPTS_TESTE = [
    "estilo_brasilia, fachada curva de concreto branco do Congresso Nacional ao entardecer",
    "estilo_brasilia, Catedral Metropolitana com coroa de colunas brancas contra o ceu azul",
    "estilo_brasilia, Palacio do Planalto com colunas curvas e espelho d'agua ao entardecer",
    "estilo_brasilia, Memorial JK com estatua sobre pilar curvo de concreto",
    "estilo_brasilia, Teatro Nacional com painel de relevos cubicos de concreto",
    "estilo_brasilia, Ponte JK com arcos brancos entrelacados sobre o lago ao por do sol",
]

SEED = 42
REPO_CONFIG1 = "lramosc1512/lora-estilo-brasilia-config1"
REPO_CONFIG2 = "lramosc1512/lora-estilo-brasilia-config2"

## 4. Gerar as imagens (base, config1, config2)

Só roda depois que os dois LoRAs estiverem publicados no Hub. Gera 1 imagem por prompt por modelo, mesma seed em cada prompt entre os 3 modelos.

In [ ]:
imagens_geradas = {}  # {(modelo, prompt_idx): imagem_PIL}

def gerar_lote(nome_modelo):
    for i, prompt in enumerate(PROMPTS_TESTE):
        gerador = torch.Generator(device=device).manual_seed(SEED)
        imagem = pipe(prompt, negative_prompt="desfocado, deformado",
                      guidance_scale=7.5, num_inference_steps=30,
                      generator=gerador).images[0]
        imagens_geradas[(nome_modelo, i)] = imagem
        print(f"{nome_modelo} - prompt {i+1}/6 gerado")

# Base (sem LoRA)
gerar_lote("base")

# Config 1
pipe.load_lora_weights(REPO_CONFIG1)
gerar_lote("config1")
pipe.unload_lora_weights()

# Config 2
pipe.load_lora_weights(REPO_CONFIG2)
gerar_lote("config2")
pipe.unload_lora_weights()

print("Geracao concluida:", len(imagens_geradas), "imagens")

## 5. Montar a grade comparativa 6x3

In [ ]:
import matplotlib.pyplot as plt

modelos = ["base", "config1", "config2"]
fig, eixos = plt.subplots(len(PROMPTS_TESTE), len(modelos), figsize=(12, 24))

for linha, prompt in enumerate(PROMPTS_TESTE):
    for coluna, modelo in enumerate(modelos):
        eixo = eixos[linha][coluna]
        eixo.imshow(imagens_geradas[(modelo, linha)])
        eixo.axis("off")
        if linha == 0:
            eixo.set_title(modelo)

plt.tight_layout()
plt.savefig("grade_comparativa.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo em grade_comparativa.png")

## 6. CLIPScore

Alinhamento imagem-texto de cada geração, média por modelo. Score mais alto = imagem mais fiel ao prompt (não mede fidelidade ao estilo visual em si, só a semântica geral).

In [ ]:
from torchmetrics.multimodal.clip_score import CLIPScore
from torchvision.transforms.functional import pil_to_tensor

metrica = CLIPScore(model_name_or_path="openai/clip-vit-base-patch16").to(device)

scores_por_modelo = {"base": [], "config1": [], "config2": []}

for (modelo, idx), imagem in imagens_geradas.items():
    tensor_img = pil_to_tensor(imagem).to(device)
    score = metrica(tensor_img, PROMPTS_TESTE[idx])
    scores_por_modelo[modelo].append(score.item())

for modelo, scores in scores_por_modelo.items():
    media = sum(scores) / len(scores)
    print(f"{modelo}: CLIPScore medio = {media:.2f}  (por prompt: {[round(s,1) for s in scores]})")

## 7. Verificação de memorização

Gera imagens com legendas quase idênticas às do dataset de treino e compara lado a lado com a imagem original — se saírem praticamente iguais, é sinal de overfitting. Troca `MODELO_PARA_TESTAR` pra checar o outro config também.

In [ ]:
import random
from pathlib import Path
from PIL import Image

MODELO_PARA_TESTAR = REPO_CONFIG2  # troque para REPO_CONFIG1 pra testar o outro

with open("/content/atelie-generativo-LeonidIA/dados/legendas.txt", "r", encoding="utf-8") as f:
    linhas = [l.strip() for l in f if l.strip()]

random.seed(7)
amostra = random.sample(linhas, 10)

pipe.load_lora_weights(MODELO_PARA_TESTAR)

fig, eixos = plt.subplots(10, 2, figsize=(6, 30))
for i, linha in enumerate(amostra):
    arquivo, legenda = linha.split(",", 1)
    legenda = legenda.strip()

    original = Image.open(f"/content/atelie-generativo-LeonidIA/dados/imagens/{arquivo}")
    gerada = pipe(legenda, num_inference_steps=30, guidance_scale=7.5).images[0]

    eixos[i][0].imshow(original)
    eixos[i][0].set_title("original" if i == 0 else "")
    eixos[i][0].axis("off")
    eixos[i][1].imshow(gerada)
    eixos[i][1].set_title("gerada" if i == 0 else "")
    eixos[i][1].axis("off")

pipe.unload_lora_weights()
plt.tight_layout()
plt.savefig("verificacao_memorizacao.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Preparar o formulário cego

Embaralha as 18 imagens da grade com nomes genéricos (sem revelar qual modelo gerou cada uma) e salva uma chave de resposta só pra você. As pessoas avaliando só veem `imagem_01.png`, `imagem_02.png`...

In [ ]:
import shutil
import csv
import random

PASTA_CEGA = Path("/content/imagens_formulario_cego")
PASTA_CEGA.mkdir(exist_ok=True)

itens = list(imagens_geradas.items())
random.seed(123)
random.shuffle(itens)

chave_resposta = []
for i, ((modelo, idx), imagem) in enumerate(itens, start=1):
    nome_anonimo = f"imagem_{i:02d}.png"
    imagem.save(PASTA_CEGA / nome_anonimo)
    chave_resposta.append({
        "arquivo": nome_anonimo,
        "modelo": modelo,
        "prompt": PROMPTS_TESTE[idx],
    })

with open(PASTA_CEGA / "CHAVE_RESPOSTA_NAO_COMPARTILHAR.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["arquivo", "modelo", "prompt"])
    writer.writeheader()
    writer.writerows(chave_resposta)

shutil.make_archive("/content/imagens_formulario_cego", "zip", PASTA_CEGA)

from google.colab import files
files.download("/content/imagens_formulario_cego.zip")
print("Baixe o zip - contem as 18 imagens anonimas + a chave de resposta (nao suba a chave no formulario!)")

## 9. Montar o formulário (manual, fora do Colab)

Google Forms não tem integração automática — isso aqui é feito manualmente:

1. Crie um Google Forms novo
2. Pra cada uma das 18 imagens: sobe a imagem + pergunta de escala 1-5 pra fidelidade ao estilo, e outra pra qualidade geral
3. Manda o link pra pelo menos 5 pessoas de fora do projeto
4. **Não mostre a `CHAVE_RESPOSTA_NAO_COMPARTILHAR.csv` pra ninguém que for responder** - isso quebra o formulário cego
5. Depois que as respostas chegarem, usa a chave pra cruzar qual nota foi pra qual modelo

## Depois disso

- Grade comparativa, CLIPScore e verificação de memorização já rodam sem esperar ninguém
- Manda o formulário cego assim que tiver as 18 imagens - quanto antes, mais cedo as respostas chegam
- Guarda `grade_comparativa.png` e `verificacao_memorizacao.png` pro relatório

```bash
git add notebooks/03_avaliacao.ipynb
git commit -m "notebook avaliacao"
git push
```